In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/xyz2005/concept-labels/valid_concepts.csv
/kaggle/input/datasets/xyz2005/concept-labels/test_concepts.csv
/kaggle/input/datasets/xyz2005/concept-labels/train_concepts.csv
/kaggle/input/datasets/xyz2005/notebook-3/train_features (1).pt
/kaggle/input/datasets/xyz2005/notebook-3/train_features.pt
/kaggle/input/datasets/xyz2005/notebook-3/test_features.pt
/kaggle/input/datasets/xyz2005/notebook-3/valid_features.pt
/kaggle/input/datasets/xyz2005/notebook-3/test_reports.csv
/kaggle/input/datasets/xyz2005/notebook-3/valid_reports.csv
/kaggle/input/datasets/xyz2005/notebook-3/train_reports.csv
/kaggle/input/datasets/xyz2005/notebook-5/training_history.csv
/kaggle/input/datasets/xyz2005/notebook-5/concept_classifier.pth
/kaggle/input/datasets/xyz2005/train-test-data/valid.csv
/kaggle/input/datasets/xyz2005/train-test-data/train.csv
/kaggle/input/datasets/xyz2005/train-test-data/test.csv


In [2]:
!pip uninstall -y transformers peft accelerate trl bitsandbytes -q

!pip install -q transformers==4.46.3
!pip install -q peft==0.13.2
!pip install -q accelerate==1.1.1
!pip install -q trl==0.12.2
!pip install -q bitsandbytes==0.44.1
!pip install -q datasets sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 14.0 MB/s eta 0:00:00


In [3]:
!pip install -U bitsandbytes==0.46.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 26.4 MB/s eta 0:00:00
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.44.1
    Uninstalling bitsandbytes-0.44.1:
      Successfully uninstalled bitsandbytes-0.44.1


In [4]:
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
train_df = pd.read_csv("/kaggle/input/datasets/xyz2005/concept-labels/train_concepts.csv")
valid_df = pd.read_csv("/kaggle/input/datasets/xyz2005/concept-labels/valid_concepts.csv")

print(train_df.head())

                                              report  Pneumonia  Cardiomegaly  \
0  low lung volumes. heart size and mediastinal c...          0             0   
1  stable cardiomediastinal silhouette. no focal ...          0             0   
2  cardiomediastinal silhouette is normal in size...          0             0   
3  normal cardiac size and contour unremarkable m...          0             0   
4  the heart is normal in size. the mediastinum i...          0             0   

   Pleural Effusion  Atelectasis  Edema  Pneumothorax  Consolidation  \
0                 1            0      0             1              1   
1                 1            0      0             1              0   
2                 1            0      0             1              0   
3                 1            0      0             1              0   
4                 0            0      0             0              0   

   Lung Opacity  Nodule  Mass  
0             0       0     0  
1             1 

In [7]:
LABELS = [
    "Pneumonia",
    "Cardiomegaly",
    "Pleural Effusion",
    "Atelectasis",
    "Edema",
    "Pneumothorax",
    "Consolidation",
    "Lung Opacity",
    "Nodule",
    "Mass"
]

def create_prompt(row):

    findings=[]

    for disease in LABELS:
        if row[disease]==1:
            findings.append(disease)

    if len(findings)==0:
        findings.append("No significant abnormality")

    findings="\n".join([f"- {x}" for x in findings])

    prompt=f"""### System
You are an expert radiologist specializing in chest X-ray interpretation.

### User
Generate a professional radiology report from the following findings.

Findings:
{findings}

### Assistant
{row["report"]}"""

    return prompt

In [8]:
train_df["text"]=train_df.apply(create_prompt,axis=1)
valid_df["text"]=valid_df.apply(create_prompt,axis=1)

In [9]:
train_dataset = Dataset.from_pandas(train_df[["text"]])
valid_dataset = Dataset.from_pandas(valid_df[["text"]])

In [10]:
MAX_LENGTH = 512

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/2311 [00:00<?, ? examples/s]

Map:   0%|          | 0/494 [00:00<?, ? examples/s]

In [11]:
train_dataset.set_format("torch")
valid_dataset.set_format("torch")

In [12]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True

)

In [13]:
model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    device_map="auto",

    trust_remote_code=True

)

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [14]:
model = prepare_model_for_kbit_training(model)

In [15]:
lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

In [16]:
model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [17]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [18]:
training_args = TrainingArguments(

    output_dir="/kaggle/working/lora_output",

    overwrite_output_dir=True,

    num_train_epochs=5,

    learning_rate=2e-4,

    per_device_train_batch_size=2,

    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=20,

    save_total_limit=2,

    fp16=True,

    report_to="none",

    load_best_model_at_end=True
)

In [19]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    tokenizer=tokenizer,

    data_collator=data_collator
)

/tmp/ipykernel_24/376308892.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [20]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being ke

Epoch,Training Loss,Validation Loss
1,0.693800,0.610328
2,0.515400,0.558348
3,0.439200,0.544410
4,0.321700,0.572805
5,0.242700,0.629003


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=1445, training_loss=0.46121459304255186, metrics={'train_runtime': 7795.3048, 'train_samples_per_second': 1.482, 'train_steps_per_second': 0.185, 'total_flos': 4.716855127965696e+16, 'train_loss': 0.46121459304255186, 'epoch': 5.0})

In [21]:
model.save_pretrained("/kaggle/working/qwen_lora")

tokenizer.save_pretrained("/kaggle/working/qwen_lora")

('/kaggle/working/qwen_lora/tokenizer_config.json',
 '/kaggle/working/qwen_lora/special_tokens_map.json',
 '/kaggle/working/qwen_lora/vocab.json',
 '/kaggle/working/qwen_lora/merges.txt',
 '/kaggle/working/qwen_lora/added_tokens.json',
 '/kaggle/working/qwen_lora/tokenizer.json')

In [22]:
import os

print(os.listdir("/kaggle/working/qwen_lora"))

['tokenizer_config.json', 'adapter_config.json', 'special_tokens_map.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'adapter_model.safetensors', 'README.md', 'added_tokens.json']
